# 1. Load dataloaders

In [ ]:
from data.dataloader import create_dataloaders

# train: linear & MEL_Scale
train_dataloader, val_dataloader = create_dataloaders(
    healthy_dir= "../../project_datasets/audio/spectogram/MEL_Scale/Healthy/",
    pd_dir= "../../project_datasets/audio/spectogram/MEL_Scale/Parkinson/",
    
    img_size=(512, 512),
    batch_size= 16,
)

In [ ]:
train_inception, val_inception = create_dataloaders(
    healthy_dir= "../../project_datasets/audio/spectogram/MEL_Scale/Healthy/",
    pd_dir= "../../project_datasets/audio/spectogram/MEL_Scale/Parkinson/",
    
    img_size=(512, 512), # minimum input size for inceptionV3 model
    batch_size= 16,
)

# 2. Load models

- **DenseNet121** – ~8M parameters  
- **EfficientNet-B0** – ~5.3M parameters  
- **InceptionV3** – ~24M parameters
- **MobileNetV3-Small** – ~2.9M parameters  
- **ResNet18** – ~11.7M parameters  

In [ ]:
from Models import (
    densenet121,
    efficientnetB0,
    InceptionV3,
    mobilenetV3,
    resnet18
)

models = [
    densenet121.DenseNet121Binary(),
    efficientnetB0.EfficientNetB0Binary(),
    InceptionV3.InceptionV3Binary(),
    mobilenetV3.MobileNetV3SmallBinary(),
    resnet18.ResNet18Binary()
]

model_names = [
    "DenseNet121",
    "EfficientNet-B0",
    "InceptionV3",
    "MobileNetV3-Small",
    "ResNet18"
]

# 3. Train models

In [ ]:
from training.trainer import train

for model, model_name in zip(models, model_names):
    train(
        model= model,
        train_dataloader= train_inception if model_name=="InceptionV3" else train_dataloader,
        val_dataloader= val_inception if model_name=="InceptionV3" else val_dataloader,
        
        model_name= model_name,
        run_name= "MEL_Scale/"+model_name,
        
        checkpoint_dir = "checkpoints/MEL_Scale",
        
        epochs= 20
    )

In [ ]:
from twilio.rest import Client
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

# Example: access a variable
account_sid = os.getenv("SID")
auth_token = os.getenv("AUTH_CODE")
from_number = os.getenv("FROM_NUMBER")
to_number = os.getenv("TO_NUMBER")
content = os.getenv("CONTENT")

client = Client(account_sid, auth_token)

message = client.messages.create(
    from_=from_number,
    content_sid=content,
    content_variables='{"1":"12/1","2":"3pm"}',
    to=to_number
)

In [ ]:
# !tensorboard --logdir=runs